### 导入必要的库

In [37]:
# 标准库导入
import os
import re
import gc
import logging
from functools import lru_cache
from typing import List, Tuple, Dict, Optional, Any

import warnings
warnings.filterwarnings("ignore",category=UserWarning)

# 第三方库导入
import pandas as pd
import torch
import jieba
import spacy
from pydantic import Field

# Gradio
import gradio as gr

# Transformers
from transformers import (
    pipeline,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# LangChain 核心
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever

# LangChain 社区
from langchain_community.document_loaders import TextLoader
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS

# LangChain 文本分割
from langchain_text_splitters import RecursiveCharacterTextSplitter

# # LangChain 向量存储
# from langchain_chroma import Chroma

# LangChain HuggingFace
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

# LangChain 经典组件
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_core.prompts import ChatPromptTemplate
# from langchain_classic.chains import RetrievalQA
# from langchain_classic.prompts import (
#     PromptTemplate,
#     ChatPromptTemplate,
#     SystemMessagePromptTemplate,
#     HumanMessagePromptTemplate,
# )
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

# Sentence Transformers
from sentence_transformers import CrossEncoder

# 本地工具
from utils import *

import dotenv
dotenv.load_dotenv()

# # 日志配置
# logger = logging.getLogger()
# logger.setLevel(logging.INFO)

True

In [2]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'


### 加载并测试qwen3-4B离线LLM

In [3]:
# model_name = os.getenv('Qwen3.5:2B','Qwen/Qwen3.5-2B')
model_name = os.getenv('Qwen3:4B','Qwen/Qwen3-4B-Instruct-2507')
model,tokenizer = load_model(model_name)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 459.64it/s]


In [4]:
model_kwargs = dict(max_new_tokens=2048,
                    temperature=1.0, top_p=0.95, 
                    top_k=20, min_p=0.0, 
                    presence_penalty=2.0, 
                    repetition_penalty=1.0)

In [5]:
messages = [{'role': 'system', 'content': '你是一位关键信息提取专家，能够提取出文本所涉及的关键人物和关键事件'},
{'role': 'user', 'content': '此日午间，薛姨妈母女两个与林黛玉等正在王夫人房里大家吃东西呢，凤姐儿得便回王夫人道：“自从玉钏儿姐姐死了，太太跟前少着一个人。太太或看准了那个丫头好，就吩咐，下月好发放月钱的。”王夫人听了，想了一想，道：“依我说，什么是例，必定四个五个的，够使就罢了，竟可以免了罢。”凤姐笑道：“论理，太太说的也是。这原是旧例，别人屋里还有两个呢，太太倒不按例了。况且省下一两银子也有限。”王夫人听了，又想一想，道：“也罢，这个分例只管关了来，不用补人，就把这一两银子给他妹妹玉钏儿罢。他姐姐伏侍了我一场，没个好结果，剩下他妹妹跟着我，吃个双分子也不为过逾了。”凤姐答应着，回头找玉钏儿，笑道：“大喜，大喜。”玉钏儿过来磕了头。王夫人问道：“正要问你，如今赵姨娘周姨娘的月例多少？”凤姐道：“那是定例，每人二两。赵姨娘有环兄弟的二两，共是四两，另外四串钱。”王夫人道：“可都按数给他们？”凤姐见问的奇怪， 忙道：“怎么不按数给！”王夫人道：“前儿我恍惚听见有人抱怨，说短了一吊钱，是什么原故？”凤姐忙笑道：“姨娘们的丫头，月例原是人各一吊。从旧年他们外头商议的， 姨娘们每位的丫头分例减半，人各五百钱，每位两个丫头，所以短了一吊钱。这也抱怨不着我，我倒乐得给他们呢，他们外头又扣着，难道我添上不成。这个事我不过是接手儿，怎么来，怎么去，由不得我作主。我倒说了两三回，仍旧添上这两分的。请总结关键人物和关键事件，按照以格式下输出（总共输出两行）：关键人物：\t 核心事件：\t'}]
ans = generate_word(model,tokenizer,messages,
                    **model_kwargs)

print(ans)

[transformers] The following generation flags are not valid and may be ignored: ['top_p', 'min_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


关键人物：王夫人、凤姐儿、玉钏儿、赵姨娘、周姨娘  
核心事件：王夫人询问姨娘们月例是否按数发放，凤姐解释因旧例调整，丫头月例由一吊钱减至五百钱，导致有人抱怨短钱，凤姐表示已多次督促补足，但实际由外头扣减，自己无法干预；王夫人决定取消丫头月例分例，将省下的银两给予玉钏儿的妹妹，以示体恤。


In [6]:
gc.collect()
torch.cuda.empty_cache()

### 文本预处理

文本读取

In [7]:
##查看文件编码
# import chardet
dataset_folder_path = f'dataset{os.sep}四大名著'
# txt_path = os.path.join(dataset_folder_path,'红楼梦.txt')
# with open(txt_path, 'rb') as f:
#     raw_data = f.read()
# result = chardet.detect(raw_data)
# print(result['encoding']) 
new_txt_path = os.path.join(dataset_folder_path,'新红楼梦.txt')
# with open(new_txt_path,'w',encoding='gbk') as f:
#     f.write(raw_data.decode('gbk',errors='ignore'))
# 将新红楼梦.txt文件内容读取出来
loader = TextLoader(new_txt_path, encoding='gbk') 
documents = loader.load()
raw_text = documents[0].page_content

文本清洗

In [8]:
#@save
def clean_raw_text(raw_text):
    # 移除【】及其内部内容
    text = re.sub(r'【.*?】', '', raw_text)
    # 移除()及其内部内容
    text = re.sub(r'\([^)]*\)', '', text)
    # 移除[]及其内部内容
    text = re.sub(r'\[.*?\]', '', text)
    # 将全角空格替换为半角空格
    text = text.replace('\u3000', ' ')
    # 合并多个换行为单个换行
    text = re.sub(r'\n+', '\n', text)
    # 合并多个空格为单个空格
    text = re.sub(r' +', ' ', text)
    # 替换常见的错误符号
    text = text.replace('> >', '》')
    text = text.replace('< <', '《')
    text = text.replace('>>', '》')
    text = text.replace('<<', '《')
    # 移除特殊控制字符
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    # 移除零宽字符
    text = re.sub(r'[\u200b-\u200f\u2028\u2029\ufeff]', '', text)
    # 统一中英文标点符号（可选：将英文标点转为中文）
    text = text.replace('(', '（').replace(')', '）')
    text = text.replace(',', '，').replace('.', '。')
    text = text.replace(':', '：').replace(';', '；')
    text = text.replace('!', '！').replace('?', '？')
    # 去除首尾空白字符
    text = text.strip()
    return text

In [9]:
cleaned_text = clean_raw_text(raw_text)
cleaned_text[:600]

'作品：红楼梦\n 作者：清·曹雪芹\n 内容简介：\n 《红楼梦》，又称《石头记》，被认为是中国最具文学成就的古典小说，是中国长篇小说创作的巅峰之作。本书第一章中说这个故事真正作者已不可考，而由曹雪芹传抄、批阅及增删数次而成，但经过学者的索隐考证后，认为曹雪芹就是本书真正的作者。\n 正文\n 第一回 甄士隐梦幻识通灵 贾雨村风尘怀闺秀\n 此开卷第一回也。作者自云：因曾历过一番梦幻之后，故将真事隐去，而借“通灵”之说，撰此《石头记》一书也。故曰“甄士隐”云云。但书中所记何事何人？自又云：“今风尘碌碌，一事无成，忽念及当日所有之女子，一一细考较去，觉其行止见识，皆出于我之上。何我堂堂须眉，诚不若彼裙钗哉？实愧则有余，悔又无益之大无可如何之日也！当此，则自欲将已往所赖天恩祖德，锦衣纨绔之时，饫甘餍肥之日，背父兄教育之恩，负师友规谈之德，以至今日一技无成，半生潦倒之罪，编述一集，以告天下人：我之罪固不免，然闺阁中本自历历有人，万不可因我之不肖，自护己短，一并使其泯灭也。虽今日之茅椽蓬牖，瓦灶绳床，其晨夕风露，阶柳庭花，亦未有妨我之襟怀笔墨者。虽我未学，下笔无文，又何妨用假语村言，敷演出一段故事来，亦可使闺阁昭传，复可悦世之目，破人愁闷，不亦宜乎？”故曰“贾雨村”云云。\n 此回中凡用“梦”用“幻”等字，是提醒阅者眼目，亦是此书立意本旨。\n 列位看官：你道此书从何而来？说起根由虽近荒唐，细按则深有趣味'

按照章节拆分RAG片段

- 先按照章节拆分
- 每个章节再按照句子拆分
- 每个句子再按照段落拆分


In [10]:
#运行前执行：uv run spacy download zh_core_web_sm
nlp = spacy.load("zh_core_web_sm")
nlp.disable_pipes("tagger", "parser", "ner")
nlp.enable_pipe("senter")
def spacy_sentence_splitter(text: str) -> List[str]:
    """
    Spacy分句函数，处理《红楼梦》对话标点，避免对话被拆分
    """
    doc = nlp(text)
    sentences = []
    temp_sent = ""
    for sent in doc.sents:
        sent_text = sent.text.strip()
        # 处理未闭合引号：合并被错误拆分的对话句
        if temp_sent and (temp_sent.count("“") > temp_sent.count("”")):
            temp_sent += sent_text
        else:
            if temp_sent:
                sentences.append(temp_sent)
            temp_sent = sent_text
    # 加入最后一个句子
    if temp_sent:
        sentences.append(temp_sent)
    # 过滤无效短句（如“正是。”“且说。”）
    sentences = [s for s in sentences if len(s) > 5]
    return sentences

In [11]:
CONFIG = {
    'chunk_size': 800,    # 单片段长度
    'chunk_overlap': 150,  # 片段重叠
    'min_len': 100         # 最小有效片段长度
}

def clean_text(text: str)->str:
    pattern = re.compile(r'[\u4e00-\u9fa5\u3002\uff1b\uff0c\uff1a\u201c\u201d\uff01\uff1f\u3001\u300a\u300b\n]+')
    cleaned_text = ''.join(pattern.findall(text))
    return cleaned_text

def split_chapters(text: str)->dict:
    titles = extract_chapter_num(text)
    chapters = []
    for title in titles:
        text_list = text.split(title)
        if len(text_list) ==2:
            chapters.extend([clean_text(text_list[0].strip('\n 正文'))])
            text = text_list[1]
    else:
        chapters.extend([clean_text(text)])
    titles = ['作品简介'] + [title.strip() for title in titles]
    return dict(zip(titles,chapters))     

def extract_chapter_num(text:str)->List:
    titles  = re.findall(r'\n 第.*回 .*\n',text)
    titles = [' '.join(title.strip().split(' ')[:3]) for title in titles]
    return titles

In [12]:
extract_chapter_num(cleaned_text[:15000])

['第一回 甄士隐梦幻识通灵 贾雨村风尘怀闺秀', '第二回 贾夫人仙逝扬州城 冷子兴演说荣国府', '第三回 贾雨村夤缘复旧职 林黛玉抛父进京都']

In [13]:
honglou_separators = ['\n',"。", "！", "？",";"]
def text2chunks(text:str,**kwargs)->List[str]:
    text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=kwargs.get('chunk_size',CONFIG['chunk_size']),  
    chunk_overlap=kwargs.get('chunk_overlap',CONFIG['chunk_overlap']),  
    separators=honglou_separators,
    length_function=len
    )
    chunks = text_splitter.split_text(text)
    return chunks

def filter_similar_texts(chunks: List[str]) -> List[str]:
    filtered_result = []
    for text in chunks:
        is_redundant = False
        for saved_text in filtered_result:
            if text in saved_text or saved_text in text:
                is_redundant = True
                if len(text) > len(saved_text):
                    filtered_result.remove(saved_text)
                    filtered_result.append(text)
                break
        if not is_redundant:
            filtered_result.append(text)
    return filtered_result

In [14]:
def build_rag_docs(text: str)->list[Document]:
    chapters = split_chapters(text)
    logging.info(f'拆分出 {len(chapters)} 个章节（简介+正文）')
    docs = []
    idx = 0
    for chapter_title, chapter_content in chapters.items():
        # 每个章节先按句子拆分，再按段落拆分
        sentences = spacy_sentence_splitter(chapter_content)
        paragraph = " ".join(sentences)
        content_chunks = text2chunks(paragraph)
        # content_chunks = text2chunks(chapter_content)
        content_chunks  = filter_similar_texts(content_chunks)
        for chunk in content_chunks:
            docs.append(Document(
                page_content= chunk,  # 仅存正文
                metadata={
                    'file_path': new_txt_path,
                    'chapter_title': chapter_title,  # 标题单独作为元数据
                    'chunk_index': idx
                }
            ))
            idx += 1
    logging.info(f'最终生成 {len(docs)} 个有效RAG片段')
    return docs

In [15]:
docs =  build_rag_docs(cleaned_text)

### 将文本转换成向量并且保存到FAISS数据库

加载[Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)离线模型,将document对象转换成向量

In [16]:
vectors_db_path = f'model{os.sep}vectors_db'
os.makedirs(vectors_db_path, exist_ok=True)
embedding_model_name = os.getenv('Qwen3-Embedding:0.6B', 'Qwen/Qwen3-Embedding-0.6B')
    
def build_rag_vector_db(docs: list[Document])->FAISS:
    # 初始化阿里Qwen-Embedding模型
    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model_name,
        model_kwargs={'device': DEVICE},  
        encode_kwargs={'normalize_embeddings': True} ,
    )
    # Document转阿里模型向量 + 存入FAISS
    vector_db = FAISS.from_documents(docs, embeddings)
    vector_db.save_local(os.path.join(vectors_db_path, 'hongloumeng_qwen_vector_db'))  # 保存为阿里模型的向量库
    ## 本地存储向量库,Chroma特别慢
    # vector_db = Chroma.from_documents(
    #     documents=docs,
    #     embedding=embeddings,
    #     persist_directory=os.path.join(vectors_db_path, 'hongloumeng_qwen_vector_db')
    # )
    logging.info('阿里Qwen-Embedding-0.6B模型的RAG向量库创建完成')
    return vector_db

In [17]:
vector_db  = None
if len(os.listdir(vectors_db_path)) == 0:
    vector_db = build_rag_vector_db(docs)
    gc.collect()
    torch.cuda.empty_cache()

加载本地embedding向量及查询

In [18]:
@lru_cache(maxsize=2)
def load_rag_vector_db(db_folder_path: str = 'hongloumeng_qwen_vector_db')->FAISS:
    """加载本地已保存的FAISS向量库（替代Chroma的load_local）"""
    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model_name,
        model_kwargs={'device': DEVICE},
        encode_kwargs={'normalize_embeddings': True}
    )
    
    vector_db_path = os.path.join(vectors_db_path, db_folder_path)
    if not os.path.exists(vector_db_path):
        raise FileNotFoundError(f"向量库路径不存在：{vector_db_path}（请先执行build_rag_vector_db创建）")
    
    vector_db = FAISS.load_local(
        embeddings=embeddings,
        folder_path=vector_db_path,
        allow_dangerous_deserialization=True,
    )
    
    logging.info("本地FAISS向量库加载完成")
    return vector_db

In [19]:
if vector_db is None: vector_db = load_rag_vector_db()

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 21979.75it/s]


### 构建检索器

In [20]:
# 检索参数（核心调优项）
RERANKER_MODEL_PATH = os.getenv('baai-bge-reranker:large', 'baai/bge-reranker-large')
#RERANKER_MODEL_PATH = 'tomaarsen/Qwen3-Reranker-0.6B-seq-cls'#'Qwen/Qwen3-Reranker-0.6B'

In [21]:
# STOP_WORDS = {"的", "了", "之", "乎", "者", "也", "且", "则", "若", "因", "便", "就", "和", "与"}
def jieba_tokenizer(text: str) -> list[str]:
    words = jieba.lcut(text)
    return [w.lower() for w in words if len(w) > 1 and not w.isspace()]

In [22]:
from functools import lru_cache
from typing import List, Optional, Dict, Any
import torch

def build_retriever(
    docs: List[Document],
    device: str = None,
    vector_top_k: int = 30,
    bm25_top_k: int = 30,
    rerank_top_k: int = 5,
    score_threshold: float = 0.9,
    ensemble_weights: List[float] = [0.7, 0.3],
    bm25_params: Dict[str, float] = None
) -> "OptimizedRetriever":
    """
    优化后的检索器构建函数
    
    优化点：
    - 模型预加载（避免重复加载）
    - Retriever 预创建（避免重复创建）
    - LRU 缓存启用
    - 参数可配置化
    
    Args:
        docs: 预处理后的文档片段
        device: 运行设备（自动识别）
        vector_top_k: 向量检索召回数量
        bm25_top_k: BM25 检索召回数量
        rerank_top_k: 重排后保留数量
        score_threshold: 重排分数阈值
        ensemble_weights: 混合检索权重 [向量权重, BM25权重]
        bm25_params: BM25 参数 {'k1': 1.5, 'b': 0.8}
    """
    
    # ==================== 1. BM25 参数配置 ====================
    if bm25_params is None:
        bm25_params = {'k1': 1.5, 'b': 0.8}
    
    # ==================== 2. 预加载模型（只加载一次）====================
    rerank_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_PATH)
    light_reranker = CrossEncoder(
        RERANKER_MODEL_PATH,
        device= device or DEVICE,
    )
    
    # ==================== 3. 预创建 Retriever（避免重复创建）====================
    vector_retriever = vector_db.as_retriever(
        search_kwargs={"k": vector_top_k}
    )
    
    bm25_retriever = BM25Retriever.from_documents(
        docs,
        k1=bm25_params['k1'],
        b=bm25_params['b'],
        tokenizer=tokenizer
    )
    bm25_retriever.k = bm25_top_k
    
    ensemble_retriever = EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=ensemble_weights,
        ensemble_type="rank",
    )
    
    # ==================== 5. 核心检索函数（带缓存）====================
    @lru_cache(maxsize=1000)
    def _cached_retrieval(
        query: str,
        rerank_topk: int = rerank_top_k,
        score_thresh: float = score_threshold
    ) -> List[Document]:
        """带缓存的检索流程"""
        # 步骤1：混合检索
        mixed_docs = ensemble_retriever.invoke(query)
        if not mixed_docs:
            logging.info("混合检索未召回相关文档")
            return []
        
        # 步骤2：多轮重排
        reranked_docs, sorted_scores, need_refine = _multi_stage_reranker(
            query, mixed_docs, rerank_topk, score_thresh
        )
        
        # 步骤3：最终过滤
        final_docs = []
        for idx, doc in enumerate(reranked_docs, 1):
            if len(doc.page_content) >= CONFIG['min_len']:
                doc.page_content = f"【文本片段{idx}】：{doc.page_content}"
                final_docs.append(doc)
        
        logging.info(f"检索完成：召回{len(mixed_docs)}条，保留{len(final_docs)}条")
        return final_docs
    
    def _multi_stage_reranker(
        query: str, 
        docs: List[Document], 
        rerank_topk: int, 
        score_thresh: float
    ) -> tuple:
        """多轮重排序"""
        if not docs:
            return [], [], False
        
        # 计算重排分数
        light_scores = light_reranker.predict(
            [(query, doc.page_content) for doc in docs]
        )
        
        # 阈值过滤
        filtered_docs = [
            (doc, score) for doc, score in zip(docs, light_scores)
            if score >= score_thresh
        ]
        
        need_refine = len(filtered_docs) == 0
        if need_refine:
            filtered_docs = list(zip(docs, light_scores))
        
        # 排序
        sorted_pairs = sorted(filtered_docs, key=lambda x: x[1], reverse=True)
        sorted_docs = [doc for doc, _ in sorted_pairs]
        
        return sorted_docs[:rerank_topk], [score for _, score in sorted_pairs], \
               need_refine
    
    # ==================== 6. LangChain 兼容封装 ====================
    class OptimizedRetriever(BaseRetriever):
        cached_retrieval_func: Any
        
        def _get_relevant_documents(self, query: str, **kwargs) -> List[Document]:
            return self.cached_retrieval_func(query, **kwargs)
    
    return OptimizedRetriever(cached_retrieval_func=_cached_retrieval)

In [23]:
# 自定义参数
retriever = build_retriever(
    docs,
    device="mps",  # 指定设备
    vector_top_k=10,
    bm25_top_k=10,
    rerank_top_k=4,
    # ensemble_weights=[0.6, 0.4],  # 调整混合权重
    # bm25_params={'k1': 1.8, 'b': 0.9}  # 自定义 BM25 参数
)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 20438.96it/s]


### 构建查询链

In [ ]:
def get_qa_chain(model:str,tokenizer:str,model_kwargs:dict={}):

    sys_prompt = (
        "你是文本信息提取专家，请根据提供的【文本片段n】（其中n表示为某个数字），按后续的要求，找到符合问题的关键信息。"
        "最终答案简洁明了，格控制字数范围在1~200字以内。输出格式为：```markdown\n……\n```"
        "Context:{context}"
    ) 
    #- 输出最终答案后，就应该执行
    custom_prompt = ChatPromptTemplate.from_messages([
        ("system", sys_prompt),
        ("human", "{input}"),
    ])
   
   
    # 2. 用transformers的pipeline封装成文本生成管道
    text_gen_pipeline = pipeline(
        model = model,
        tokenizer = tokenizer,
        task="text-generation",
    )
    
    # 3. 转换成LangChain兼容的LLM对象（这是关键：变成Runnable类型）
    llm = HuggingFacePipeline(pipeline=text_gen_pipeline,model_kwargs=model_kwargs)
    
    question_answer_chain = create_stuff_documents_chain(llm, custom_prompt)
    qa_chain = create_retrieval_chain(retriever, question_answer_chain)
    return qa_chain

In [ ]:
qa_chain =  get_qa_chain(model,tokenizer,model_kwargs)


In [ ]:
def extract_retriever_result(result:dict)->str:
    result = result['answer']
    result = re.search(r'Human:.*?```markdown\n(.*?)\n```', result,re.DOTALL)
    if result is None:
        return '没有找到相关答案'
    result = result.group(1)
    result = result.strip()
    result = result.replace('\\n','')
    return result


In [27]:
dataset_folder = 'dataset/四大名著/hongloumeng/'

In [28]:
test_data = load_data(os.path.join(dataset_folder,'test_public_simple.json'))
test_df = pd.DataFrame(test_data)
test_df.head()

,query,golden_passage_index,answer
0,大观园题咏是谁在宫中编排的？,23,贾元春
1,贾政是贾宝玉的什么亲属？,2,父亲
2,大观园中的女尼、爱茶且性情清高者是谁？,41,妙玉
3,荣国府内众姐妹居住游赏的园林名为何？,30,大观园
4,与林黛玉并称的另一位女主角是谁？,8,薛宝钗


In [29]:
res_rag = []
# for idx,query in enumerate(queries[1:2],1): 
for idx,query in enumerate(test_df['query'].values,1):  
    result = qa_chain.invoke({'input':query})
    res_rag.append(extract_retriever_result(result))
    gc.collect()
    torch.cuda.empty_cache()


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_len

In [30]:
df = test_df.copy(deep=True)
df['Qwen3:4B-RAG'] = res_rag

In [32]:
normal_res = []
for idx,query in enumerate(test_df['query'].values,1):
    messages = [{'role': 'system', 'content': '你是一位精通中国四大名著的专家，可以准确回答我关于四大名著的问题,将最终输出答案输出控制在250字以内。'}]
    ans = generate_word(model,tokenizer,messages,**model_kwargs)
    messages.extend([{'role': 'assistant', 'content': ans},{'role': 'user', 'content': query}])
    res = generate_word(model,tokenizer,messages,**model_kwargs)
    normal_res.extend([res])
    gc.collect()
    torch.cuda.empty_cache()

In [36]:
df['Qwen3:4B-Normal'] = normal_res
df.to_excel(os.path.join(dataset_folder,'hongloumeng_QA.xlsx'),sheet_name='Qwen3-4B',index=False)


In [ ]:
character_for_chatbot = '贾装懂'
prompt_for_roleplay = '你是一位精通中国四大名著的专家，可以准确回答我关于四大名著的问题,将最终输出答案字数严格控制在200字以内。'

# function to clear the coversation
def reset() -> List:
    return interact_roleplay([], prompt_for_roleplay,running_mode='Normal'),'Normal'

# function to call the model to generate the response
def interact_roleplay(chatbot:List[Dict[str, str]], user_input: str, temp=1.0, running_mode='Normal') -> List[Tuple[str, str]]:
    '''
    * Arguments

      - user_input: the user input of each round of conversation

      - temp: the temperature parameter of this model. Temperature is used to control the output of the chatbot.
              The higher the temperature is, the more creative response you will get.

    '''
    # print(running_mode)
    try:
        messages = []
        messages.extend(chatbot)
        messages.append({'role': 'user', 'content': user_input+'将最终输出答案字数严格控制在200字以内'})
        if  running_mode == 'Normal':
            content = generate_word(model,tokenizer,messages,temperature=temp,max_new_tokens=2048)
        else:
            content = extract_retriever_result(qa_chain.invoke({'input':user_input}))
        chatbot.extend([{'role':'user','content':user_input},{'role':'assistant','content':content}])
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'Error occurred: {e}')
        chatbot.extend([{'role':'user','content':user_input},{'role':'assistant','content':f'Sorry, an error occurred: {e}'}])
    return chatbot

def export_roleplay(chatbot: List[Tuple[str, str]], description: str) -> None:
    '''
    * Arguments

      - chatbot: the model itself, the conversation is stored in list of tuples

      - description: the description of this task

    '''
    target = {'chatbot': chatbot, 'description': description}
    with open('part2.json', 'w') as file:
        json.dump(target, file)



first_dialogue = interact_roleplay([], prompt_for_roleplay)

# This part constructs the Gradio UI interface
with gr.Blocks() as demo:
    gr.Markdown(f'# 四大名著问答')
    chatbot = gr.Chatbot(value = first_dialogue,avatar_images=('Images/567.png','Images/jdb.png'),height=800)
    description_textbox = gr.Textbox(label=f'The character the bot is playing', interactive = False, value=f'{character_for_chatbot}')
    input_textbox = gr.Textbox(label='Input', value = '')
    with gr.Column():
        gr.Markdown('#  Temperature\n Temperature is used to control the output of the chatbot. The higher the temperature is, the more creative response you will get.')
        temperature_slider = gr.Slider(0.0, 1.0, 0.7, step = 0.1, label='Temperature')
    with gr.Row():
        sent_button = gr.Button(value='Send',)
        reset_button = gr.Button(value='Reset')
        export_button = gr.Button(value='Export')
        dropdown = gr.Dropdown(choices=["Normal", "RAG"],label="模式",value="Normal")
    sent_button.click(interact_roleplay, inputs=[chatbot, input_textbox, temperature_slider,dropdown], outputs=[chatbot])
    reset_button.click(reset, outputs=[chatbot,dropdown])
    export_button.click(export_roleplay, inputs=[chatbot, description_textbox])


demo.launch(debug = True, server_name="0.0.0.0", share=True)